In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
%cd drive/MyDrive/graph-fake-news-detection

/content/drive/MyDrive/graph-fake-news-detection


In [ ]:
import torch

TORCH, DEVICE = str(torch.__version__).split('+')

!pip install torch_geometric
!pip install pyg_lib torch_scatter torch_sparse -f https://data.pyg.org/whl/torch-{TORCH}+{DEVICE}.html

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 42.2 MB/s eta 0:00:00
Looking in links: https://data.pyg.org/whl/torch-2.11.0+cu128.html
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.1/5.1 MB 35.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 100.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 96.9 MB/s eta 0:00:00


In [ ]:
%matplotlib inline

from pathlib import Path
import pandas as pd

DATASET = "gossipcop"
FEATURE = "bert"
EPOCHS = 100
PATIENCE = 10
SEEDS = "123 456 777 101 112"
DEVICE_STR = "cpu" if DEVICE == "cpu" else "cuda:0"

RESULTS_DIR = Path("results")

# Giá trị dự phòng (dùng khi file kết quả HPO thiếu cột tương ứng)
DEFAULTS = {"nhid": 128, "batch_size": 128, "lr": 1e-3, "wd": 1e-4}


def load_best_params(model_name):
    """Đọc tham số tốt nhất từ kết quả HPO (ưu tiên file riêng của model)."""
    per_model = RESULTS_DIR / model_name / DATASET / FEATURE / f"{model_name}_best.csv"
    summary = RESULTS_DIR / "best_params_summary.csv"

    if per_model.exists():
        row = pd.read_csv(per_model).iloc[0]
    elif summary.exists():
        df = pd.read_csv(summary)
        df = df[(df["model"] == model_name) & (df["dataset"] == DATASET)]
        if df.empty:
            raise FileNotFoundError(
                f"Không có dòng {model_name}/{DATASET} trong {summary}. Hãy chạy HPO trước."
            )
        row = df.iloc[0]
    else:
        raise FileNotFoundError(
            f"Không thấy {per_model} hoặc {summary}. Hãy chạy HPO trước."
        )

    best = dict(DEFAULTS)
    for key in DEFAULTS:
        if key in row.index and pd.notna(row[key]):
            best[key] = row[key]

    return {
        "nhid": int(best["nhid"]),
        "batch_size": int(best["batch_size"]),
        "lr": float(best["lr"]),
        "wd": float(best["wd"]),
    }


def find_flag(script_path, candidates):
    """Trả về tên tham số đầu tiên mà script có khai báo, hoặc None."""
    text = Path(script_path).read_text(encoding="utf-8")
    for flag in candidates:
        if flag in text:
            return flag
    return None


def run_best(model_name, script, extra_args="", hidden_flag="--nhid"):
    """Chạy script huấn luyện cuối cùng với tham số tốt nhất từ HPO."""
    best = load_best_params(model_name)
    script_path = f"src/models/{script}.py"
    print(f"[{model_name} / {DATASET}] best params: {best}")

    parts = [
        f"--feature {FEATURE}",
        f"--epochs {EPOCHS}",
        f"--patience {PATIENCE}",
        f"--lr {best['lr']}",
        f"{hidden_flag} {best['nhid']}",
        f"--batch_size {best['batch_size']}",
        f"--dataset {DATASET}",
        f"--device {DEVICE_STR}",
        f"--seeds {SEEDS}",
    ]
    if extra_args:
        parts.append(extra_args)

    wd_flag = find_flag(script_path, ["--weight_decay", "--wd"])
    if wd_flag:
        parts.append(f"{wd_flag} {best['wd']}")
    else:
        print(f"WARNING: {script_path} không có tham số weight decay "
              f"-> wd={best['wd']} KHÔNG được áp dụng.")

    cmd = f"{script_path} " + " ".join(parts)
    print("%run", cmd)
    get_ipython().run_line_magic("run", cmd)

In [ ]:
run_best("gnn", "gnn", extra_args="--model sage --concat")

In [ ]:
run_best("gcnfn", "gcnfn", extra_args="--concat")

In [ ]:
run_best("bigcn", "bigcn", extra_args="--TDdroprate 0.2 --BUdroprate 0.2")

In [ ]:
run_best("gnncl", "gnncl", hidden_flag="--hidden")

In [ ]:
%matplotlib inline

%run src/compare_all.py \
    --dataset 'gossipcop' \
    --feature 'bert'


COMBINED RESULTS
   model  accuracy_mean  f1_mean  precision_mean  recall_mean  auc_mean  ap_mean
GNN-SAGE         0.9686   0.9688          0.9609       0.9770    0.9935   0.9932
   GCNFN         0.9551   0.9550          0.9545       0.9556    0.9850   0.9836
   BiGCN         0.9154   0.9161          0.9080       0.9247    0.9697   0.9694
  GNN-CL         0.9628   0.9629          0.9571       0.9690    0.9866   0.9828

Saved to: /content/drive/MyDrive/graph-fake-news-detection/results/combined/gossipcop/bert


<Figure size 640x480 with 0 Axes>